# Why You Need Polar Decomposition: Fast Differentiable SVD on GPU


### 1. Clone the repository

In [ ]:
!git clone https://github.com/anonymous-502475/cans_svd.git

### 2. Install Dependencies

In [ ]:
!pip install -r cans_svd/requirements.txt

In [ ]:
%cd cans_svd

### 3. Import JAX and CANS-SVD

In [ ]:
import jax
import jax.test_util
import jax.numpy as jnp
from jax import config
config.update("jax_default_matmul_precision", "float32")
jax.config.update("jax_enable_x64", True)

from src import svd_pseudo, svd_tikhonov

In [ ]:
def generate_matrix(n: int, m: int, key) -> jnp.array:
    """
    Generate a random matrix.
    """
    return jax.random.normal(key=key, shape=(n, m), dtype=jnp.float64)

In [ ]:
def f_pseudo(A: jnp.array) -> float:
    """
    Pre-defined loss function for pseudo-inverse based differentiation of SVD.
    """
    U, S, Vt = svd_pseudo(A)
    return jnp.sum(jnp.abs(U)) + jnp.sum(jnp.abs(S)) + jnp.sum(jnp.abs(Vt))


def f_tikhonov(A: jnp.array) -> float:
    """
    Pre-defined loss function for Tikhonov regularized based differentiation of SVD.
    """
    U, S, Vt = svd_tikhonov(A)
    return jnp.sum(jnp.abs(U)) + jnp.sum(jnp.abs(S)) + jnp.sum(jnp.abs(Vt))

### 4. Check the accuracy of algorithms on an ill-conditioned matrix

First, lets generate random matrix:

In [ ]:
A = generate_matrix(10, 10, key=jax.random.PRNGKey(0))

Now we can check gradients using finite-differences

In [ ]:
jax.test_util.check_grads(
    f_pseudo,
    (A,),
    order=1,
    modes=["rev"],
    eps=1e-6,
    atol=1e-4,
    rtol=1e-4,
)

In [ ]:
jax.test_util.check_grads(
    f_tikhonov,
    (A,),
    order=1,
    modes=["rev"],
    eps=1e-6,
    atol=1e-4,
    rtol=1e-4,
)